In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [2]:
train_data = pd.read_csv('../data/model/train.csv')
train_labels = train_data['redemption_status'].values
train_data = train_data.drop(['id','redemption_status'], axis=1)
valid_data = pd.read_csv('../data/model/valid.csv')
valid_labels = valid_data['redemption_status'].values
valid_data = valid_data.drop(['id','redemption_status'], axis=1)

In [3]:
params = {}
params['label'] = train_labels
params['categorical_feature'] = ['customer_id']
params['feature_name'] = list(train_data.columns)
train_matrix = lgb.Dataset(train_data.values, **params)
params = {}
params['label'] = valid_labels
params['categorical_feature'] = ['customer_id']
params['feature_name'] = list(valid_data.columns)
valid_matrix = lgb.Dataset(valid_data.values, **params)

In [4]:
booster = {}
booster['boosting_type'] = 'gbdt'
booster['objective'] = 'binary'
booster['learning_rate'] = 0.01
booster['num_leaves'] = 48
booster['max_depth'] = 6
booster['max_bin'] = 256
booster['subsample'] = 0.5
booster['subsample_freq'] = 1
booster['colsample_bylevel'] = 0.5
booster['colsample_bytree'] = 0.5
booster['min_split_gain'] = 0.0
booster['min_sum_hessian'] = 1
booster['nthread'] = 3
booster['verbose'] = 0
booster['metric'] = 'auc'

In [5]:
params = {}
params['params'] = booster
params['train_set'] = train_matrix
params['valid_sets'] = [train_matrix, valid_matrix]
params['num_boost_round'] = 2000
params['early_stopping_rounds'] = 200
params['verbose_eval'] = 25

In [6]:
model = lgb.train(**params)

/home/ubuntu/anaconda3/lib/python3.7/site-packages/lightgbm/basic.py:1243: UserWarning: Using categorical_feature in Dataset.
  warnings.warn('Using categorical_feature in Dataset.')


Training until validation scores don't improve for 200 rounds
[25]	training's auc: 0.985778	valid_1's auc: 0.952667
[50]	training's auc: 0.990302	valid_1's auc: 0.95633
[75]	training's auc: 0.992552	valid_1's auc: 0.957775
[100]	training's auc: 0.993971	valid_1's auc: 0.958011
[125]	training's auc: 0.994912	valid_1's auc: 0.958443
[150]	training's auc: 0.995685	valid_1's auc: 0.957627
[175]	training's auc: 0.996409	valid_1's auc: 0.956725
[200]	training's auc: 0.996847	valid_1's auc: 0.957288
[225]	training's auc: 0.997252	valid_1's auc: 0.957572
[250]	training's auc: 0.997594	valid_1's auc: 0.957715
[275]	training's auc: 0.997882	valid_1's auc: 0.957859
[300]	training's auc: 0.998151	valid_1's auc: 0.95851
[325]	training's auc: 0.998375	valid_1's auc: 0.958516
[350]	training's auc: 0.998586	valid_1's auc: 0.958444
[375]	training's auc: 0.998783	valid_1's auc: 0.95786
[400]	training's auc: 0.998924	valid_1's auc: 0.957694
[425]	training's auc: 0.999077	valid_1's auc: 0.9571
[450]	train

In [7]:
model.save_model('../data/model/lightgbm_v2.model')

In [8]:
importance = model.feature_importance(importance_type='gain')
importance = pd.DataFrame(importance, columns=['importance'])
importance['feature'] = list(valid_data.columns)
importance['importance'] = importance['importance'] / importance['importance'].max()
importance = importance[['feature', 'importance']]
importance = importance.sort_values(by='importance', ascending=False)
importance = importance.reset_index(drop=True)

In [9]:
importance.head(10)

,feature,importance
0,customer_id,1.000000
1,max_trx_cust_coup_price,0.350104
2,cust_cdsc_cnt,0.310617
3,cust_cdsc_sum,0.298309
4,cnt_coup_cdsc,0.262215
5,sum_trx_cust_coup_price,0.240487
6,cust_coup_prc,0.217792
7,cust_cdsc,0.196726
8,sum_coup_cdsc,0.189108
9,sum_trx_cust_coup_dsc,0.188529
